In [1]:
%pip install pypdf
%pip install sentence-transformers
%pip install qdrant-client
%pip install langchain langchain-experimental langchain-community

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
     |████████████████████████████████| 1.0 MB 3.2 MB/s eta 0:00:01
     |████████████████████████████████| 207 kB 9.9 MB/s eta 0:00:01
     |████████████████████████████████| 2.3 MB 8.3 MB/s eta 0:00:01
     |████████████████████████████████| 311 kB 28.1 MB/s eta 0:00:01
     |████████████████████████████████| 397 kB 8.8 MB/s eta 0:00:011
     |████████████████████████████████| 3.3 MB 2.7 MB/s eta 0:00:01
     |████████████████████████████████| 1.3 MB 9.4 MB/s eta 0:00:011
     |████████████████████████████████| 130 kB 8.5 MB/s eta 0:00:01
     |████████████████████████████████| 54 kB 8.5 MB/s  eta 0:00:01
     |████████████████████████████████| 650 kB 5.8 MB/s eta 0:00:01
     |████████████████████████████████| 319 kB 7.7 MB/s eta 0:00:01
     |████████████████████████████████| 129 kB 8.3 

In [7]:
# PDF RAG using Sentence Transformers + Qdrant (No OpenAI)

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
)
from langchain_experimental.text_splitter import SemanticChunker

# ============================================================
# Configuration
# ============================================================

PDF_PATH = "/home/linux00/Downloads/Dhanush_Resume_Photo.pdf"
COLLECTION_NAME = "resume_rag"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100
TOP_K = 3

# ============================================================
# PDF Reader
# ============================================================

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract text from a PDF file."""

    reader = PdfReader(pdf_path)

    content = ""

    for page in reader.pages:
        text = page.extract_text()

        if text:
            content += text + "\n"

    return content


# ============================================================
# Text Chunking
# ============================================================

def chunk_text(
    text: str,
    chunk_size: int = 1000,
    overlap: int = 100
):
    """Split text into overlapping chunks."""

    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


# ============================================================
# Embedding Model
# ============================================================

def load_embedding_model():
    """Load SentenceTransformer model."""

    return SentenceTransformer(EMBEDDING_MODEL)


# ============================================================
# Qdrant Setup
# ============================================================

def create_collection(client, vector_size):
    """Create fresh Qdrant collection."""

    try:
        client.delete_collection(COLLECTION_NAME)
        print(f"Deleted existing collection: {COLLECTION_NAME}")
    except Exception:
        pass

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=vector_size,
            distance=Distance.COSINE,
        ),
    )

    print(f"Created collection: {COLLECTION_NAME}")


# ============================================================
# Store Embeddings
# ============================================================

def store_chunks(client, chunks, embeddings):
    """Insert chunks and embeddings into Qdrant."""

    points = []

    for idx, (chunk, vector) in enumerate(
        zip(chunks, embeddings)
    ):
        points.append(
            PointStruct(
                id=idx,
                vector=vector.tolist(),
                payload={
                    "text": chunk
                },
            )
        )

    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points,
    )

    print(f"Inserted {len(points)} chunks into Qdrant")


# ============================================================
# Search Function
# ============================================================

def search(client, model, query, top_k=3):
    """Search similar chunks from Qdrant."""

    query_vector = model.encode(query)

    results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector.tolist(),
        limit=top_k,
    )

    return results


# ============================================================
# Main
# ============================================================

def main():

    print("\nReading PDF...")

    content = extract_text_from_pdf(PDF_PATH)

    print(f"Total Characters : {len(content)}")

    print("\nChunking text...")

    chunks = chunk_text(
        content,
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP,
    )

    model = load_embedding_model()

    print(f"Total Chunks : {len(chunks)}")

    print("\nLoading embedding model...")


    print("Generating embeddings...")

    embeddings = model.encode(
        chunks,
        show_progress_bar=True,
    )

    print("Embeddings generated")

    print("\nConnecting to Qdrant...")

    client = QdrantClient(
        host=QDRANT_HOST,
        port=QDRANT_PORT,
    )

    create_collection(
        client,
        vector_size=len(embeddings[0]),
    )

    store_chunks(
        client,
        chunks,
        embeddings,
    )

    print("\nRAG System Ready")


    query = "Does he knows typescript"

    results = search(
        client,
        model,
        query,
        top_k=TOP_K,
    )

    print("\nTop Results")

    for rank, hit in enumerate(results, start=1):

        print("\n" + "=" * 80)

        print(f"Rank  : {rank}")
        print(f"Score : {hit.score:.4f}")

        print("\nRetrieved Chunk:\n")

        print(hit.payload)

        print("=" * 80)


if __name__ == "__main__":
    main()



Reading PDF...
Total Characters : 5415

Chunking text...
Total Chunks : 7

Loading embedding model...
Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated

Connecting to Qdrant...
Deleted existing collection: resume_rag
Created collection: resume_rag
Inserted 7 chunks into Qdrant

RAG System Ready

Top Results

Rank  : 1
Score : 0.3482

Retrieved Chunk:

{'text': 'Dhanushvarman J\n dhanushvarmanj@gmail.com /ne+91 9944705776\n LinkedIn /gtbGitHub\n Chennai, India\nSUMMARY\nA highly skilled and results driven Frontend Developer with 2+ years of experience specializing in the React ecosystem. Proven\nability to build and maintain scalable applications, with expertise in implementing advanced features such as Progressive Web\nApps (PW As) with offline support, real-time data visualization, and automated PDF reporting. Adept at leading projects and\ncollaborating directly with clients to deliver high-quality, impactful user experiences.\nSKILLS\nFrontend Development:React JS, Redux, Redux-Saga, React Native, Ant Design, Bootstrap, Tailwind CSS, Formik\nProgramming & Web:JavaScript (ES6+), HTML5, CSS3 and Node.js\nAPI Inte